In [ ]:
from local_paths import RAW_DATA_PATH, MASTER_DATA_PATH #gitignored local path
from repo_paths import *

# Reading PREVIEW data files

In [ ]:
from preview_study import get_intakes
intakes = get_intakes(RAW_DATA_PATH)

# Calculating ASEP

In [ ]:
INTAKE_COLS = [
    'subject_id',
    'code',
    'fao_subgroup_code',
    'animal_proportion',
    'ENERJ',
    'ENERJ_animal'
]

PARTICIPANT_COLS = [
    'v01_agecon',
    'Sex',
    'Educational Attainment Level',
    'cm_intensity',
    'var26', #smoking status 0,1,2
    'v02_mvpa', #physical activity minutes per day
    'v02_b_hba1c_mean', #glycated hemoglobin
    'BMI',
    'fr_days',
    'ASEP',
    'ENERJ',
    'ENERJ_animal',
    'ENERJ_per_d',
    'PROT_per_d',
    'PROT_g_per_BW',
    'FIBC_per_d',
    'NA_per_d',
    'CHOL_per_d',
    'CHO_QR', #carbohydrate quality ratio
    'FIBC_per_MJ',
    'CHOL_per_MJ',
    'NA_per_MJ',
    'ep-CHO',
    'ep-FIBC',
    'ep-FAMS',
    'ep-FAPU',
    'ep-FASAT',
    'ep-FATRN',
    'ep-FAT',
    'ep-PROT',
    'ep-SUGAR',
    'ep-SUCS', #sucrose
    'ep-ALC',
    'ALC_per_d', #adding alcohol consumption in grams
    'vegetarian',
    'v02_fs_kol_hdl_mean',
    'v02_fs_kol_ldl_mean',
    'v02_fs_kol_mean',
    'v02_fs_trigly_mean',
    'veg_g'
]

In [ ]:
import pandas as pd
food_properties = pd.read_csv(
    DATA_PATH+'/food_properties.csv',
    index_col='code',
    dtype={
        'animal_proportion':float,
        'has_meat_or_poultry': bool,
        'has_fish':bool,
        'has_seafood':bool,
        'ingredient_subgroup_code': 'Int64',
        'fao_subgroup_code': 'Int64'
    }
)

In [ ]:

#calculating animal source energy proportion
intakes_with_props = intakes.merge(
    food_properties,
    left_on='code',
    right_index=True,
    how='left'
)
intakes_with_props['ENERJ_animal'] = intakes_with_props.ENERJ * intakes_with_props.animal_proportion

#missing food_properties
missing = intakes_with_props[intakes_with_props.animal_proportion.isna()][['Code','code','Name']].value_counts().to_frame().reset_index()

#food codes found in records, that do not have an entry in the property database
#get written into a template file that is manually filled in with the food properties

food_props_to_fill = missing[['code','Name']].copy()
food_props_to_fill.columns = ['code','name']
for c in ['animal_proportion','has_meat_or_poultry','has_fish','has_seafood']:
    food_props_to_fill[c] = None
food_props_to_fill.to_csv(f'{DATA_PATH}/food_properties_new.csv', index=False)

In [ ]:
intakes_per_person = intakes_with_props.drop(
    columns=[
        'IvSocSec',
        'PdPeriod',
        'DaDay',
        'MaMeal',
        'MaType',
        'CoRow',
        'CoIcAdId1',
        'CoIcAdId2',
        'MainGroup',
        'SubGroup',
        'CoFormel',
    ]
).groupby('subject_id').sum(numeric_only=True)
intakes_per_person['ASEP'] = intakes_per_person.ENERJ_animal / intakes_per_person.ENERJ
intakes_per_person['PUFA_per_fat'] = intakes_per_person.FAPU / intakes_per_person.FAT

In [ ]:
veggies = intakes_with_props.query('899 < fao_subgroup_code < 1000',engine='python')[['subject_id','CoWeight']].groupby('subject_id').sum()
veggies.columns = ['veg_g']
intakes_per_person = intakes_per_person.join(veggies)

In [ ]:
#picking up the number of food record days per each subject and adding it to the df
fr_days = intakes_with_props.groupby('subject_id')['date'].nunique()
fr_days.name = 'fr_days'
intakes_per_person = intakes_per_person.join(fr_days)

In [ ]:
#calculating energy densities of macronutrients (kJ per 1 g of macronutrient)

e_densities = {
    'FAT': 37,
    'FAPU': 37,
    'FASAT': 37,
    'FAMS': 37,
    'FAPUN3': 37,
    'FAPUN6': 37,
    'FATRN': 37,
    'CHO': 17,
    'SUGAR': 17,
    'SUCS': 17,
    'PROT': 17,
    'FIBC': 8,
    'ALC': 29,
}

e_proportions = pd.DataFrame()

for k,v in e_densities.items():
    # add a column for the total energy from macro nutrient k
    # participants[column] contanins intake in grams, v is energy density in kJ/g, result is total energy intake
    e_proportions[f'e-{k}'] = intakes_per_person[k] * v
    # calculcate energy proportion of k, it is customary to show them in E%, hence * 100
    e_proportions[f'ep-{k}'] = e_proportions[f'e-{k}'] / intakes_per_person.ENERJ * 100

intakes_per_person = pd.concat(
    [
        intakes_per_person,
        e_proportions
    ],
    axis=1
)
intakes_per_person.index.names = ['subject_id']

In [ ]:
for i in ['ENERJ','FIBC','CHOL','NA','PROT','ALC','F18D2CN6', 'F18D3N3', 'F20D4N6', 'F20D5N3', 'F22D6N3']:
    intakes_per_person[f'{i}_per_d'] = intakes_per_person[i] / intakes_per_person.fr_days

In [ ]:
for i in ['FIBC','CHOL','NA']:
    intakes_per_person[f'{i}_per_MJ'] = intakes_per_person[i] / (intakes_per_person.ENERJ / 1000)

# Identifying participants who did not consume meat or poultry (vegetarians)

In [ ]:
intakes_per_person['vegetarian'] = (intakes_per_person['has_meat_or_poultry'] == 0).astype(int)

# Lab Results and background info

In [ ]:
from preview_study import read_preview_study_csv
lab_results = read_preview_study_csv(
    f'{RAW_DATA_PATH}/preview_datahubdata_14-May-2024_FINAL.csv',
    index_col='subject_id',
)

In [ ]:
lab_results['BMI'] = lab_results.v02_wght / lab_results.v01_hghtavg**2

In [ ]:
education = lab_results.var14

# Overriding education grouping with assesment of freetext input in var15
# REDACTED FOR PRIVACY


lab_results['Educational Attainment Level'] = education.map({
    2: 'Low', 
    3: 'Low',
    4: 'Intermediate',
    5: 'High',
    6: 'High',
    7: None,  # Drop group 7
})

# Medication 

In [ ]:
#importing df containing reported medication filled-up by a nurse

meds = read_preview_study_csv(
    f'{RAW_DATA_PATH}/PREVIEW_CID1_concomitant_medication_OpenClinica.csv',
    index_col='subject_id',
    parse_dates=['cm_start','cm_stop','v02_date']
)

In [ ]:
# ensuring cm_atc (atc-code for medication class) is treated as a string
meds['cm_atc'] = meds['cm_atc'].astype(str)

# filtering the df for lipid lowering meds, i.e. rows where cm_atc starts with 'C10'
# however, excluding ezetimibe 'C10AX09', which was used by one participant only
c10_meds = meds[meds['cm_atc'].str.startswith('C10') & (meds['cm_atc'] != 'C10AX09')].copy()

In [ ]:
# normalizing the cm_name for consistency and extracting the main medication names
def normalize_med_name(name):
    if pd.isna(name) or name.strip() == '':
        return ''  # Return an empty string for NaN or empty values
    name = name.lower().strip()
    # Removing drug brand names (within parentheses) and keeping names (text outside)
    while '(' in name:
        # Remove text from the first '(' to the matching ')'
        left = name.split('(', 1)[0].strip()
        right = name.split(')', 1)[-1].strip()
        name = f"{left} {right}".strip()
    return name.strip()

# Apply normalization to the cm_name column using .loc
c10_meds.loc[:, 'cm_name_normalized'] = c10_meds['cm_name'].apply(normalize_med_name)

In [ ]:
def classify_intensity(row):
    cm_name = row['cm_name_normalized']
    cm_dose = row['cm_dose']
    
    # Define intensity criteria as a dictionary
    intensity_rules = {
        'High': [
            (('atorvastatin', 'atrovastatin', 'atorvastatiini', 'lipitor'), 40, 80),
            (('rosuvastatin', 'rosvastatin', 'crestor'), 20, 40),
            (('simvastatin', 'simvastatiini'), 20, 40),
            (('pravastatin',), 40, 80),
            (('fluvastatine',), 40, 80)
        ],
        'Moderate': [
            (('atorvastatin', 'atrovastatin', 'atorvastatiini', 'lipitor'), 10, 20),
            (('rosuvastatin', 'rosvastatin', 'crestor'), 5, 10),
            (('simvastatin', 'simvastatiini'), 20, 40),
            (('pravastatin',), 40, 80),
            (('fluvastatine',), 40, 80)
        ],
        'Low': [
            (('pravastatin',), 10, 40),
            (('lovastatin',), 20, 40),
            (('atorvastatin',), 5, 5),
            (('fluvastatine',), 20, 40),
            (('simvastatin',), 5, 10)
        ]
    }
    
    # Handle missing dose - classify as Moderate if name matches simvastatin
    if pd.isna(cm_dose) and cm_name in ('simvastatin', 'simvastatiini'):
        return 'Moderate'
    
    # Check each intensity level
    for intensity, rules in intensity_rules.items():
        for names, min_dose, max_dose in rules:
            if cm_name in names and min_dose <= cm_dose <= max_dose:
                return intensity
    
    return 'Unknown'

c10_meds['cm_intensity'] = c10_meds.apply(classify_intensity, axis=1)

In [ ]:
c10_meds_subset = c10_meds[['cm_intensity', 'cm_name', 'cm_atc', 'cm_dose']]

In [ ]:
#one participant had reported earlier and later dosages of statin medication, considering only the earlier 
# (the one present at the time of food record collection)
c10_meds_subset = c10_meds_subset[~c10_meds_subset.index.duplicated(keep='first')]

# Data on physical activity

In [ ]:
#importing df containing accelerometer data
phys_act = read_preview_study_csv(
    f'{RAW_DATA_PATH}/PREVIEW_CID1_accelerometer_data_06-09-2024.csv',
    index_col='subject_id',
)

# Joining tables

In [ ]:
participants = intakes_per_person.join(lab_results, how='left')

In [ ]:
#calculating carbohydrate quality ratio (CQR) as follows: fibre total intake(g)/carbohydrates total intake (g)
participants['CHO_QR'] = participants['FIBC'] / participants['CHO']

In [ ]:
#calculating protein intake also per body weigth
participants['PROT_g_per_BW'] = participants['PROT_per_d'] / participants['v02_wght']

In [ ]:
participants = participants.join(c10_meds_subset, how='left')

In [ ]:
participants = participants.join(phys_act, how='left')

In [ ]:
participants['Sex'] = participants.v01_gen.map({1: 'Female', 2: 'Male'})

# Sanitizing and writing output files

Only picking data columns known to be needed in analysis.

Mapping ids in study data to random integers in output files. These will change on every run of the code, they only ensure uniqueness and mapping with the code run. These ids are sufficient to run data analysis code from the sanitized output files.

In [ ]:
participants_out = participants[PARTICIPANT_COLS].copy()
intakes_out = intakes_with_props[INTAKE_COLS].copy()

import numpy as np
new_ids = np.random.permutation(len(participants.index))
id_mapping = dict(zip(participants.index, new_ids))

participants_out.index = participants.index.map(id_mapping)
intakes_out.subject_id = intakes_with_props.subject_id.map(id_mapping)

from pathlib import Path
Path(MASTER_DATA_PATH).mkdir(parents=True, exist_ok=True)
participants_out.to_csv(f'{MASTER_DATA_PATH}/{PARTICIPANTS_FNAME}.csv')
participants_out.to_excel(f'{MASTER_DATA_PATH}/{PARTICIPANTS_FNAME}.xlsx')
intakes_out.to_csv(f'{MASTER_DATA_PATH}/{INTAKES_FNAME}.csv', index=False)
len(intakes)